# Particle-to-multipole expansion (P2M)

## Purpose

P2M compresses all dipoles in one source region into Cartesian multipole
coefficients about a chosen expansion centre. It is the first algebraic step
of an upward FMM pass, although this repository does not yet implement that
tree traversal. P2M constructs a source representation; it does **not**
evaluate a field.

## Mathematical definition

For a multi-index $\alpha=(\alpha_x,\alpha_y,\alpha_z)$, the implementation
stores one $M_\alpha$ for every $|\alpha|\le p$. The dense coefficient count is

$$N_p=\binom{p+3}{3}=\frac{(p+1)(p+2)(p+3)}{6}.$$

Dipoles contribute through terms $\alpha-e_k$, so no term is valid for
$\alpha=(0,0,0)$. Therefore $M_{(0,0,0)}=0$ for pure dipole sources.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
expansion_order = 4
n_sources = 8
source_box_half_width = 0.4
random_seed = 42
expansion_centre = np.array([0.0, 0.0, 0.0])
shifted_centre = np.array([0.15, -0.10, 0.05])

## Problem setup and operator evaluation

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(
    -source_box_half_width,
    source_box_half_width,
    size=(n_sources, 3),
)
dipole_moments = rng.normal(size=(n_sources, 3))

multipole_coefficients = cdfmm.p2m_dipole(
    expansion_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)
multi_indices = cdfmm.multi_indices(expansion_order)
degrees = np.sum(multi_indices, axis=1)

expected_count = (
    (expansion_order + 1)
    * (expansion_order + 2)
    * (expansion_order + 3)
    // 6
)
print(f"Expansion order: {expansion_order}")
print(f"Coefficient count: {len(multipole_coefficients)} (expected {expected_count})")
print(f"Monopole coefficient M_(0,0,0): {multipole_coefficients[0]:.16e}")

## Coefficient ordering

In [ ]:
print(" linear     alpha      degree          M_alpha")
print("------------------------------------------------")
for linear_index, (alpha, degree, value) in enumerate(
    zip(multi_indices, degrees, multipole_coefficients)
):
    alpha_text = f"({alpha[0]:d},{alpha[1]:d},{alpha[2]:d})"
    print(f" {linear_index:5d}   {alpha_text:>9s}   {degree:5d}   {value: .8e}")

## Coefficient magnitude by degree

In [ ]:
figure, axes = plt.subplots(figsize=(8, 5))
plot_coefficients_by_degree(
    axes,
    multipole_coefficients,
    expansion_order,
    "P2M coefficient magnitudes",
)
figure.tight_layout()

## Effect of order and expansion centre

In [ ]:
print(" order   coefficient count   max |M_alpha|")
for order in range(1, 7):
    coefficients = cdfmm.p2m_dipole(
        expansion_centre,
        source_positions,
        dipole_moments,
        order=order,
    )
    print(f" {order:5d}   {len(coefficients):17d}   {np.max(np.abs(coefficients)): .6e}")

shifted_coefficients = cdfmm.p2m_dipole(
    shifted_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
plot_coefficients_by_degree(
    axes[0],
    multipole_coefficients,
    expansion_order,
    "Centre at origin",
)
plot_coefficients_by_degree(
    axes[1],
    shifted_coefficients,
    expansion_order,
    "Shifted expansion centre",
    colour="tab:orange",
)
figure.tight_layout()

## What to observe

The coefficient vector grows cubically with order but its first entry remains
exactly zero. Shifting the centre changes higher moments because source
displacements are measured relative to that centre. Those different vectors
still represent the same source cloud when translated or evaluated
consistently.